# CRITEO Uplift Modeling — Kaggle execution

**One-click entry point.** Attach a dataset containing `criteo-uplift-v2.1.csv`,
then Run All. This reproduces the full experiment: data processing, a response
baseline, T-Learner, X-Learner, Causal Forest, and the final test-set comparison.

This notebook is an **orchestrator** — every model, transform, and metric is
imported from `src/`. Nothing is reimplemented here. For the reasoning behind
each step, read the methodology notebooks:

| Notebook | Covers |
|---|---|
| `01_data_processing` | data contract, feature semantics, the split |
| `02_baseline_models` | response model, and why it is *not* a causal estimator |
| `03_uplift_models` | T-Learner, X-Learner, the fold-local leakage fix |
| `04_causal_forest` | Causal Forest, categorical encoding, final comparison |

## Run parameters

`SAMPLE_ROWS = None` runs the complete ~13.9M-row experiment.

⚠️ **Runtime warning.** The Causal Forest is the bottleneck: `econml`'s
`CausalForest` runs with `n_jobs=1` (required for reproducible predictions),
so a full-data fit can take many hours and may exceed Kaggle's session limit.
If you want a fast end-to-end check first, set `SAMPLE_ROWS = 1_000_000`
below — the sample is stratified on `(treatment, conversion)`, and **every
model sees the same rows**, so the comparison stays like-for-like.

In [ ]:
SAMPLE_ROWS = None      # None = full dataset; e.g. 1_000_000 for a fast run
SEED = 42
RUN_CAUSAL_FOREST = True   # set False to skip the slowest stage

## Section 1 — Environment setup

Locate the repository (this notebook does not assume the kernel's working
directory), make `src/` importable, and verify every dependency actually
imports before any expensive work starts.

In [ ]:
import sys, platform
from pathlib import Path


def find_repo_root() -> Path:
    bases = [Path.cwd(), *Path.cwd().parents, Path("/kaggle/working"), Path("/kaggle/input")]
    for base in bases:
        if not base.is_dir():
            continue
        if (base / "src" / "data.py").is_file():
            return base
        for child in sorted(p for p in base.iterdir() if p.is_dir()):
            if (child / "src" / "data.py").is_file():
                return child
    return None


REPO_ROOT = find_repo_root()

if REPO_ROOT is None and Path("/kaggle/working").is_dir():
    # Repo not attached -- try cloning it (requires Internet enabled in kernel settings).
    import subprocess
    target = Path("/kaggle/working/causal-uplift-modeling")
    print("Repository not found; attempting clone into", target)
    result = subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/arthur105204/causal-uplift-modeling.git", str(target)],
        capture_output=True, text=True,
    )
    print(result.stdout or result.stderr)
    REPO_ROOT = find_repo_root()

if REPO_ROOT is None:
    raise RuntimeError(
        "Could not locate the repository. Either attach it as a Kaggle dataset, "
        "clone it into /kaggle/working, or enable Internet in the kernel settings "
        "so this cell can clone it automatically."
    )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("python     :", platform.python_version())
print("repo root  :", REPO_ROOT)

In [ ]:
import importlib

REQUIRED = ["numpy", "pandas", "sklearn", "lightgbm", "econml", "pyarrow", "matplotlib", "yaml"]
missing = []
for name in REQUIRED:
    try:
        module = importlib.import_module(name)
        print(f"{name:12s} {getattr(module, '__version__', 'n/a')}")
    except ImportError as exc:
        missing.append(name)
        print(f"{name:12s} MISSING ({exc})")

if missing:
    print("\nInstalling missing packages...")
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=False)
    print("Re-run this cell to confirm. If econml is missing, install with: pip install econml==0.17.0")
else:
    print("\nAll dependencies present.")

In [ ]:
# Everything below comes from src/ -- no model code is defined in this notebook.
from src.data import (
    CATEGORICAL_FEATURES, CONTINUOUS_FEATURES, FEATURE_COLUMNS,
    PRIMARY_OUTCOME, SECONDARY_OUTCOME, TREATMENT_COLUMN,
    basic_summary, load_config, load_csv, on_kaggle, output_dir, resolve_csv_path, save_parquet,
)
from src.preprocessing import (
    CausalForestCategoricalEncoder, LightGBMFeatureTransform, train_validation_test_split,
)
from src.models import (
    fit_causal_forest, fit_response_model, fit_t_learner, fit_x_learner,
    predict, predict_causal_forest_tau,
)
from src.evaluation import compute_ate, evaluate_ranking, response_diagnostics

CONFIG = load_config()
OUTPUT_DIR = output_dir()
print("on kaggle  :", on_kaggle())
print("outputs    :", OUTPUT_DIR)
print("imports OK")

## Section 2 — Dataset setup

The dataset slug is **not** hardcoded: `resolve_csv_path()` searches every
attached `/kaggle/input/<slug>/` for `criteo-uplift-v2.1.csv`, falling back to
a local `data/raw/` copy for development.

In [ ]:
CSV_PATH = resolve_csv_path()
print("dataset path :", CSV_PATH)
print("dataset size : %.1f MB" % (CSV_PATH.stat().st_size / 1024**2))

In [ ]:
frame = load_csv(CSV_PATH)
print("shape   :", frame.shape)
print("columns :", list(frame.columns))
frame.head()

In [ ]:
# Verify the causal contract before doing anything expensive.
missing_features = [f for f in FEATURE_COLUMNS if f not in frame.columns]
assert not missing_features, f"missing features: {missing_features}"
assert TREATMENT_COLUMN in frame.columns, f"missing treatment column {TREATMENT_COLUMN!r}"
assert PRIMARY_OUTCOME in frame.columns, f"missing outcome column {PRIMARY_OUTCOME!r}"
assert set(frame[TREATMENT_COLUMN].unique()) <= {0, 1}, "treatment must be binary"
assert set(frame[PRIMARY_OUTCOME].unique()) <= {0, 1}, "conversion must be binary"

print("X  = %s .. %s  (%d features)" % (FEATURE_COLUMNS[0], FEATURE_COLUMNS[-1], len(FEATURE_COLUMNS)))
print("     continuous  :", CONTINUOUS_FEATURES)
print("     categorical :", CATEGORICAL_FEATURES)
print("T  =", TREATMENT_COLUMN)
print("Y  =", PRIMARY_OUTCOME, " (secondary:", SECONDARY_OUTCOME + ")")
print()
basic_summary(frame)

In [ ]:
ate = compute_ate(frame[TREATMENT_COLUMN], frame[PRIMARY_OUTCOME])
print("Population ATE : %.6f  (95%% CI %.6f .. %.6f)" % (ate.ate, ate.ci_95_low, ate.ci_95_high))
print("Relative lift  : %s" % (f"{ate.relative_lift:.1%}" if ate.relative_lift else "n/a"))

In [ ]:
if SAMPLE_ROWS is not None and SAMPLE_ROWS < len(frame):
    from sklearn.model_selection import train_test_split as _tts
    strata = frame[TREATMENT_COLUMN].astype(str) + "_" + frame[PRIMARY_OUTCOME].astype(str)
    frame, _ = _tts(frame, train_size=SAMPLE_ROWS, random_state=SEED, stratify=strata)
    frame = frame.reset_index(drop=True)
    print(f"Stratified subsample: {len(frame):,} rows (all models see these same rows)")
else:
    print(f"Using the full dataset: {len(frame):,} rows")

## Section 3 — Execute pipeline

Mirrors notebooks 01 → 04, calling the same `src/` functions they call.

**Split.** 70/15/15, jointly stratified on `(treatment, conversion)`. Train
fits; validation early-stops and selects; **test is scored once**, in Section 4.

In [ ]:
split_cfg = CONFIG["split"]
train_frame, val_frame, test_frame = train_validation_test_split(
    frame,
    train_fraction=split_cfg["train_fraction"],
    validation_fraction=split_cfg["validation_fraction"],
    test_fraction=split_cfg["test_fraction"],
    seed=SEED,
)
for name, part in [("train", train_frame), ("validation", val_frame), ("test", test_frame)]:
    print(f"{name:11s} {len(part):>10,}  treatment={part[TREATMENT_COLUMN].mean():.4f}  "
          f"conversion={part[PRIMARY_OUTCOME].mean():.5f}")

In [ ]:
# LightGBM feature representation: continuous stay float64; the eight
# categorical features get a TRAIN-fitted pandas categorical dtype, so
# LightGBM uses native categorical splits instead of a false ordering.
transform = LightGBMFeatureTransform().fit(train_frame)
X_train, X_val, X_test = (transform.transform(f) for f in (train_frame, val_frame, test_frame))
Y_train, Y_val, Y_test = (f[PRIMARY_OUTCOME] for f in (train_frame, val_frame, test_frame))
T_train, T_val, T_test = (f[TREATMENT_COLUMN] for f in (train_frame, val_frame, test_frame))

test_scores = {}   # model -> test-set scores, collected for the final comparison
val_metrics = {}   # model -> validation RankingMetrics
print("feature dtypes:", dict(X_train.dtypes.value_counts().items()))

### 3.1 Response baseline (non-causal comparator)

In [ ]:
import time

start = time.perf_counter()
response_model = fit_response_model(X_train, Y_train, X_val, Y_val, seed=SEED)
print(f"fitted in {time.perf_counter() - start:.1f}s  (best iteration {response_model.best_iteration})")

val_scores = predict(response_model, X_val)
print(response_diagnostics(val_scores, Y_val))
val_metrics["Response LightGBM"] = evaluate_ranking(val_scores, T_val, Y_val)
test_scores["Response LightGBM"] = predict(response_model, X_test)
print("validation qini_above_random:", round(val_metrics["Response LightGBM"].qini_above_random, 4))

### 3.2 T-Learner

In [ ]:
start = time.perf_counter()
t_learner = fit_t_learner(X_train, T_train, Y_train, X_val, T_val, Y_val, seed=SEED)
print(f"fitted in {time.perf_counter() - start:.1f}s")

val_metrics["T-Learner"] = evaluate_ranking(t_learner.predict_tau(X_val), T_val, Y_val)
test_scores["T-Learner"] = t_learner.predict_tau(X_test)
print("validation qini_above_random:", round(val_metrics["T-Learner"].qini_above_random, 4))

### 3.3 X-Learner

`fit_x_learner` takes the **raw** frame deliberately: it fits a *fold-local*
categorical transform inside each cross-fitting fold. Passing a globally-fit
frame would leak each fold's category vocabulary into the other, and the
function raises rather than let that happen silently.

In [ ]:
start = time.perf_counter()
x_learner = fit_x_learner(train_frame, T_train, Y_train, seed=SEED)
print(f"fitted in {time.perf_counter() - start:.1f}s")

val_metrics["X-Learner"] = evaluate_ranking(x_learner.predict_tau(X_val), T_val, Y_val)
test_scores["X-Learner"] = x_learner.predict_tau(X_test)
print("validation qini_above_random:", round(val_metrics["X-Learner"].qini_above_random, 4))

### 3.4 Causal Forest

`econml.grf.CausalForest` takes a dense numeric matrix and has no native
categorical support, so categorical features are frequency-capped top-K
one-hot encoded (`OTHER` bucket for the tail and for unseen categories).
Feeding raw tokens through as floats would invent an ordering the splits
would exploit — `fit_causal_forest` rejects that outright.

**This is the slow stage** (`n_jobs=1` for reproducibility).

In [ ]:
if RUN_CAUSAL_FOREST:
    encoder = CausalForestCategoricalEncoder(k=CONFIG["causal_forest"]["categorical_top_k"]).fit(train_frame)
    X_train_cf, X_val_cf, X_test_cf = (encoder.transform(f) for f in (train_frame, val_frame, test_frame))
    print("encoded features:", X_train_cf.shape[1])

    start = time.perf_counter()
    causal_forest = fit_causal_forest(X_train_cf, T_train, Y_train, seed=SEED)
    print(f"fitted in {time.perf_counter() - start:.1f}s")

    val_metrics["Causal Forest"] = evaluate_ranking(
        predict_causal_forest_tau(causal_forest, X_val_cf), T_val, Y_val
    )
    test_scores["Causal Forest"] = predict_causal_forest_tau(causal_forest, X_test_cf)
    print("validation qini_above_random:", round(val_metrics["Causal Forest"].qini_above_random, 4))
else:
    print("Causal Forest skipped (RUN_CAUSAL_FOREST = False)")

## Section 4 — Final results

Everything below is scored on the **test** partition — never fit on,
early-stopped against, or used for selection. A random ranking is included as
the honest floor: an uplift model that cannot beat it has not earned its
complexity.

In [ ]:
import numpy as np
import pandas as pd

test_metrics = {
    label: evaluate_ranking(scores, T_test, Y_test) for label, scores in test_scores.items()
}
rng = np.random.default_rng(SEED)
test_metrics["Random (reference)"] = evaluate_ranking(rng.uniform(size=len(T_test)), T_test, Y_test)

rows = []
for label, m in test_metrics.items():
    row = {"model": label, "auuc_above_random": m.auuc_above_random,
           "qini_above_random": m.qini_above_random,
           "auuc_area": m.auuc_area, "qini_area": m.qini_area}
    row.update({f"uplift@{k}": v for k, v in m.uplift_at_k.items()})
    rows.append(row)

comparison = pd.DataFrame(rows).set_index("model").sort_values("qini_above_random", ascending=False)
save_parquet(comparison.reset_index(), OUTPUT_DIR / "final_comparison.parquet")
comparison.round(5)

In [ ]:
causal_models = [m for m in ("T-Learner", "X-Learner", "Causal Forest") if m in test_metrics]
ranked = comparison.drop(index="Random (reference)", errors="ignore")
best = ranked.index[0]

print("BEST MODEL:", best)
print(f"  Qini above random : {ranked.loc[best, 'qini_above_random']:.5f}")
print(f"  AUUC above random : {ranked.loc[best, 'auuc_above_random']:.5f}")
print(f"  uplift@10pct      : {ranked.loc[best, 'uplift@10pct']}")
print()
random_qini = test_metrics["Random (reference)"].qini_above_random
beat_random = [m for m in ranked.index if ranked.loc[m, "qini_above_random"] > random_qini]
print("Models beating the random reference:", beat_random or "NONE")

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
for label, m in test_metrics.items():
    style = "--" if label.startswith("Random") else "-"
    ax1.plot(m.qini_curve["coverage"], m.qini_curve["qini_gain"], style, label=label)
    ax2.plot(m.uplift_curve["coverage"], m.uplift_curve["uplift_gain"], style, label=label)
ax1.set_xlabel("Coverage"); ax1.set_ylabel("Qini gain"); ax1.set_title("Qini curves (test)"); ax1.legend()
ax2.set_xlabel("Coverage"); ax2.set_ylabel("Uplift gain"); ax2.set_title("Uplift curves / AUUC (test)"); ax2.legend()
fig.tight_layout()

In [ ]:
comparison[["qini_above_random", "auuc_above_random"]].plot.barh(
    figsize=(9, 4), title="Test-set ranking performance above the random reference"
)

### CATE distribution

A model whose predicted effect is nearly constant is not finding heterogeneity, whatever its Qini says.

In [ ]:
cate = pd.DataFrame({label: test_scores[label] for label in causal_models})
display_stats = cate.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T
display_stats.round(6)

In [ ]:
fig, axes = plt.subplots(1, max(len(causal_models), 1), figsize=(5 * max(len(causal_models), 1), 3.5), squeeze=False)
for ax, label in zip(axes[0], causal_models):
    ax.hist(test_scores[label], bins=60)
    ax.axvline(0.0, color="k", linestyle="--", linewidth=1)
    ax.set_title(f"{label}\npredicted CATE")
fig.tight_layout()

### Uplift decile analysis

If a ranking is real, observed uplift should decline from decile 1 (highest predicted) downward.

In [ ]:
deciles = pd.DataFrame({
    label: test_metrics[label].decile_table.set_index("decile")["observed_uplift"]
    for label in causal_models
})
deciles.round(5)

In [ ]:
ax = deciles.plot(marker="o", figsize=(9, 4),
                  title="Observed test uplift by predicted-CATE decile (1 = highest predicted)")
ax.axhline(0.0, color="k", linestyle="--", linewidth=1)
ax.set_xlabel("Decile of predicted CATE"); ax.set_ylabel("Observed uplift")

In [ ]:
if len(causal_models) > 1:
    print("Spearman rank correlation between causal models' predicted CATE:")
    print(cate.corr(method="spearman").round(3))

## Conclusions

State the result in the form the evidence supports:

> Among the evaluated implementations, **<best model>** achieved the strongest
> test-set uplift ranking on CRITEO-UPLIFTv2.1 under this protocol.

What this does **not** establish:

- that the winning estimator is universally best — one dataset, one
  implementation of each method, one hyperparameter setting;
- that a meta-learner family is intrinsically superior — what is measured is a
  *specific implementation* on *these* features;
- that predicted CATE is a true individual treatment effect — both potential
  outcomes are never observed for anyone, which is also why no PEHE against
  ground truth is reported;
- that the response model's ranking is causal, however strong its AUC.

The comparison also reports point estimates only — there are no confidence
intervals on the differences between models, so a small gap between two
models should not be read as a decisive ranking.

The most informative row is usually Response LightGBM against the causal
estimators: it separates *who converts* from *who converts because of the
treatment*, which is the entire premise of uplift modeling.